# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 100151

# Grid Search de hiperparametros
# cp y num_trees_max quedan FIJOS
PARAM$rpart$cp <- -1
PARAM$num_trees_max <- 32

# Valores a recorrer en el Grid Search
# feature_fraction: proporcion de features (columnas) utilizada en cada arbol
PARAM$grid_feature_fraction <- c(0.25, 0.50, 0.75)

# hiperparametros de rpart
PARAM$grid_minsplit <- c(200, 500, 800)
PARAM$grid_minbucket_divisor <- c(2, 4, 6) # minbucket se calcula como minsplit / divisor (ej: minsplit/2, minsplit/4)
PARAM$grid_maxdepth <- c(4, 6, 8, 10)

# cada combinacion tendra su propia semilla derivada de la semilla primigenia
# para que una ejecucion interrumpida pueda retomarse de forma reproducible
PARAM$archivo_resultados <- "gridsearch_resultados.csv"

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp4020_grid_search"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [ ]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

In [ ]:
# que tamanos de ensemble grabo a disco y pusheo a Kaggle
# son las potencias de 2, hasta num_trees_max
grabar <- 2^(0:floor(log2(PARAM$num_trees_max)))
print(grabar)

In [ ]:
# Tabla de resultados: una fila por arbol terminado en cada combinacion.
# Sirve tambien para poder retomar la ejecucion si Colab se corta.
if (file.exists(PARAM$archivo_resultados)) {
  tb_grid_resultados <- fread(PARAM$archivo_resultados)
} else {
  tb_grid_resultados <- data.table(
    iter = integer(),
    feature_fraction = numeric(),
    minsplit = integer(),
    minbucket = integer(),
    maxdepth = integer(),
    cp = numeric(),
    arbolito = integer(),
    kaggle_enviado = logical(),
    archivo_kaggle = character(),
    fecha_hora = character()
  )
}

cat("Arboles registrados previamente:", nrow(tb_grid_resultados), "\n")

La semilla se establece dentro de cada combinacion del Grid Search,mpara que la corrida sea reproducible aunque se reanude tras un corte.

In [ ]:
# Grid Search: loops anidados para cada hiperparametro.
# cp = -1 y num_trees_max = 32 quedan fijos.
#
# La logica de recuperacion es:
#   - si una combinacion ya llego a 32 arboles, se saltea completa;
#   - si quedo a mitad de camino, se vuelven a construir los arboles
#     anteriores (sin volver a pushear sus submissions) y se continua.
#
# La semilla de cada combinacion depende de iter y de PARAM$semilla_primigenia,
# por lo que al reanudar se obtiene la misma secuencia de campos aleatorios.

iter <- 0
t_ini <- Sys.time()

for (vfeature_fraction in PARAM$grid_feature_fraction) {
  for (vmin_split in PARAM$grid_minsplit) {
    for (vmin_bucket_divisor in PARAM$grid_minbucket_divisor) {
      for (vmax_depth in PARAM$grid_maxdepth) {

        # minbucket como fraccion de minsplit (ej: minsplit/2, minsplit/4)
        vmin_bucket <- as.integer(vmin_split / vmin_bucket_divisor)

        iter <- iter + 1

        # identifico cuanto de esta combinacion ya fue ejecutado
        realizados <- tb_grid_resultados[
          feature_fraction == vfeature_fraction &
          minsplit == vmin_split &
          minbucket == vmin_bucket &
          maxdepth == vmax_depth &
          cp == PARAM$rpart$cp,
          max(arbolito, na.rm = TRUE)
        ]
        if (!is.finite(realizados)) realizados <- 0

        if (realizados >= PARAM$num_trees_max) {
          cat("SKIP - combinacion ya completa:",
              sprintf("ff=%.2f minsplit=%d minbucket=%d maxdepth=%d\n",
                      vfeature_fraction, vmin_split, vmin_bucket, vmax_depth))
          next
        }

        cat("\n========================================\n")
        cat(sprintf("Iteracion %d - ff=%.2f minsplit=%d minbucket=%d maxdepth=%d cp=%.1f\n",
                    iter, vfeature_fraction, vmin_split, vmin_bucket, vmax_depth, PARAM$rpart$cp))
        cat(sprintf("Ya realizados: %d / %d\n", realizados, PARAM$num_trees_max))

        # carpeta donde se guardan las predicciones de esta combinacion
        carpeta_combo <- sprintf("ff%.2f_ms%d_mb%d_md%d",
                                  vfeature_fraction, vmin_split, vmin_bucket, vmax_depth)
        dir.create(carpeta_combo, showWarnings = FALSE)

        # parametros de rpart de esta combinacion
        PARAM$rpart$cp <- -1
        PARAM$rpart$minsplit <- vmin_split
        PARAM$rpart$minbucket <- vmin_bucket
        PARAM$rpart$maxdepth <- vmax_depth

        # semilla reproducible por combinacion
        set.seed(PARAM$semilla_primigenia + iter)

        # reinicio del acumulador para esta combinacion
        tb_prediccion <- dfuture[, list(numero_de_cliente)]
        tb_prediccion[, prob_acumulada := 0]

        for (arbolito in seq(PARAM$num_trees_max)) {
          message(sprintf("Iteracion %d - arbol %d/%d", iter, arbolito, PARAM$num_trees_max))

          qty_campos_a_utilizar <- max(1L, as.integer(length(campos_buenos) * vfeature_fraction))

          # elijo los campos al azar
          campos_random <- sample(campos_buenos, qty_campos_a_utilizar)

          # paso de un vector a un string con los elementos separados por +
          campos_random <- paste(campos_random, collapse = " + ")

          # armo la formula para rpart
          formulita <- paste0("clase_ternaria ~ ", campos_random)

          # genero el arbol de decision
          modelo <- rpart(formulita,
            data = dtrain,
            xval = 0,
            control = PARAM$rpart
          )

          # aplico el modelo a los datos que no tienen clase
          prediccion <- predict(modelo, dfuture, type = "prob")
          tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

          # Solo pusheo los puntos que todavia no estaban registrados.
          # Esto evita submissions duplicadas si se reanuda una corrida.
          envio_kaggle <- FALSE
          archivo_kaggle <- ""

          if (arbolito %in% grabar && arbolito > realizados) {
            umbral_corte <- (1 / 40) * arbolito
            tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

            archivo_kaggle <- file.path(
              carpeta_combo,
              paste0(
                "KA420_",
                sprintf("ff%.2f_ms%d_mb%d_md%d_t%03d",
                        vfeature_fraction, vmin_split, vmin_bucket,
                        vmax_depth, arbolito),
                ".csv"
              )
            )

            # grabo el archivo de submission
            fwrite(
              tb_prediccion[, list(numero_de_cliente, Predicted)],
              file = archivo_kaggle,
              sep = ","
            )

            # subida a Kaggle
            comando <- "kaggle competitions submit"
            competencia <- "-c utn-2026-inicial"
            arch <- paste("-f", archivo_kaggle)
            mensaje <- paste0(
              "-m 'ff=", vfeature_fraction,
              " cp=", PARAM$rpart$cp,
              " minsplit=", PARAM$rpart$minsplit,
              " minbucket=", PARAM$rpart$minbucket,
              " maxdepth=", PARAM$rpart$maxdepth,
              " trees=", arbolito, "'"
            )
            linea <- paste(comando, competencia, arch, mensaje)
            salida <- system(linea, intern = TRUE)
            cat(salida, sep = "\n")
            envio_kaggle <- TRUE
          }

          # registro persistente: una fila por arbol NUEVO terminado.
          # Si estamos reanudando una combinacion parcial, no duplicamos
          # las filas de los arboles que ya estaban registrados.
          if (arbolito > realizados) {
            fila_resultado <- data.table(
              iter = iter,
              feature_fraction = vfeature_fraction,
              minsplit = vmin_split,
              minbucket = vmin_bucket,
              maxdepth = vmax_depth,
              cp = PARAM$rpart$cp,
              arbolito = arbolito,
              kaggle_enviado = envio_kaggle,
              archivo_kaggle = archivo_kaggle,
              fecha_hora = format(Sys.time(), "%Y-%m-%d %H:%M:%S")
            )

            archivo_existe <- file.exists(PARAM$archivo_resultados)
            fwrite(
              fila_resultado,
              file = PARAM$archivo_resultados,
              sep = ",",
              append = archivo_existe,
              col.names = !archivo_existe
            )
            tb_grid_resultados <- rbindlist(list(tb_grid_resultados, fila_resultado), fill = TRUE)
          }
        }
      }
    }
  }
}

t_fin <- Sys.time()
cat("Tiempo total de ejecucion: ", t_fin - t_ini, "\n")



---

